In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
deposits_size = pd.read_csv('../int/deposits_size.csv')
deposits_category = pd.read_csv('../int/deposits_category.csv')
deposits_pool = pd.read_csv('../int/deposits_pool.csv')
staked_pool_size = pd.read_csv('../int/active_validators_size.csv')
staked_category = pd.read_csv('../int/active_validators_category.csv')
staked_pool = pd.read_csv('../int/active_validators_pool.csv')

deposits_size = deposits_size[deposits_size['slot'] >= 6206400]
deposits_category = deposits_category[deposits_category['slot'] >= 6206400]
deposits_pool = deposits_pool[deposits_pool['slot'] >= 6206400]

In [3]:
staked_pool_size.loc[:, staked_pool_size.columns != 'slot'] *= 32
staked_category.loc[:, staked_category.columns != 'slot'] *= 32
staked_pool.loc[:, staked_pool.columns != 'slot'] *= 32

In [4]:
rewards_size = pd.read_csv('../int/rewards_size.csv').drop(columns=['epoch'])
rewards_category = pd.read_csv('../int/rewards_category.csv').drop(columns=['epoch'])
rewards_pool = pd.read_csv('../int/rewards_pool.csv').drop(columns=['epoch'])
rewards_size = rewards_size[rewards_size['slot'].isin(deposits_size['slot'])]
rewards_category = rewards_category[rewards_category['slot'].isin(deposits_category['slot'])]
rewards_pool = rewards_pool[rewards_pool['slot'].isin(deposits_pool['slot'])]

rewards_size = rewards_size.drop(columns=('total'))
rewards_category = rewards_category.drop(columns=('total'))
rewards_pool = rewards_pool.drop(columns=('total'))

rewards_size['total'] = rewards_size.drop(columns=['slot']).mean(axis=1)
rewards_category['total'] = rewards_category.drop(columns=['slot']).mean(axis=1)
rewards_pool['total'] = rewards_pool.drop(columns=['slot']).mean(axis=1)

In [5]:
deposits_size_set = deposits_size
deposits_size_set['slot'] = deposits_size_set['slot'] // 7200 * 7200

# Group by the day and get the last slot of each day and sum values for all columns
deposits_size_set = deposits_size_set.groupby('slot').agg(
    {
        'slot': 'last',
        '1': 'sum',
        '2-5': 'sum',
        '6-19': 'sum',
        '20-99': 'sum',
        '100+': 'sum',
        'total': 'sum'
    }
).reset_index(drop=True)

deposits_size_set


,slot,1,2-5,6-19,20-99,100+,total
0,6206400,64.0,128.0,512.0,5184.0,14720.0,20608.0
1,6213600,192.0,768.0,640.0,3520.0,66800.0,71920.0
2,6220800,768.0,896.0,2240.0,11040.0,106816.0,121760.0
3,6228000,448.0,704.0,1920.0,4256.0,50320.0,57648.0
4,6235200,256.0,864.0,2720.0,5248.0,85296.0,94384.0
...,...,...,...,...,...,...,...
382,8956800,224.0,448.0,320.0,192.0,32268.0,33452.0
383,8964000,160.0,480.0,64.0,256.0,17014.0,17974.0
384,8971200,1248.0,224.0,960.0,4384.0,9730.0,16546.0
385,8978400,192.0,384.0,64.0,192.0,7203.0,8035.0


In [6]:
deposits_category_set = deposits_category
deposits_category_set['slot'] = deposits_category_set['slot'] // 7200 * 7200

# Create a dictionary for aggregation
agg_dict = {col: 'sum' for col in deposits_category_set.columns if col != 'slot'}
agg_dict['slot'] = 'last'

# Group by 'slot' and apply the aggregation
deposits_category_set = deposits_category_set.groupby('slot').agg(agg_dict).reset_index(drop=True)

deposits_category_set

,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total,slot
0,4544.0,0.0,5600.0,224.0,576.0,9664.0,20608.0,6206400
1,9920.0,0.0,2928.0,192.0,2976.0,55904.0,71920.0,6213600
2,36608.0,0.0,28800.0,1888.0,3392.0,51072.0,121760.0,6220800
3,27616.0,0.0,13104.0,1344.0,2176.0,13408.0,57648.0,6228000
4,29984.0,0.0,13072.0,6976.0,1440.0,42912.0,94384.0,6235200
...,...,...,...,...,...,...,...,...
382,2592.0,23440.0,316.0,64.0,576.0,6464.0,33452.0,8956800
383,704.0,1200.0,902.0,32.0,8352.0,6784.0,17974.0,8964000
384,3488.0,768.0,226.0,0.0,1056.0,11008.0,16546.0,8971200
385,2048.0,352.0,1027.0,32.0,448.0,4128.0,8035.0,8978400


In [7]:
deposits_category_set

,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total,slot
0,4544.0,0.0,5600.0,224.0,576.0,9664.0,20608.0,6206400
1,9920.0,0.0,2928.0,192.0,2976.0,55904.0,71920.0,6213600
2,36608.0,0.0,28800.0,1888.0,3392.0,51072.0,121760.0,6220800
3,27616.0,0.0,13104.0,1344.0,2176.0,13408.0,57648.0,6228000
4,29984.0,0.0,13072.0,6976.0,1440.0,42912.0,94384.0,6235200
...,...,...,...,...,...,...,...,...
382,2592.0,23440.0,316.0,64.0,576.0,6464.0,33452.0,8956800
383,704.0,1200.0,902.0,32.0,8352.0,6784.0,17974.0,8964000
384,3488.0,768.0,226.0,0.0,1056.0,11008.0,16546.0,8971200
385,2048.0,352.0,1027.0,32.0,448.0,4128.0,8035.0,8978400


In [8]:
deposits_pool_set = deposits_pool
deposits_pool_set['slot'] = deposits_pool_set['slot'] // 7200 * 7200

# Create a dictionary for aggregation
agg_dict = {col: 'sum' for col in deposits_pool_set.columns if col != 'slot'}
agg_dict['slot'] = 'last'

# Group by 'slot' and apply the aggregation
deposits_pool_set = deposits_pool_set.groupby('slot').agg(agg_dict).reset_index(drop=True)

deposits_pool_set

,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total,slot
0,0.0,0.0,3072.0,0.0,672.0,576.0,4512.0,0.0,416.0,10592.0,768.0,20608.0,6206400
1,704.0,0.0,2208.0,0.0,0.0,2912.0,928.0,0.0,3200.0,61856.0,112.0,71920.0,6213600
2,1184.0,0.0,22336.0,0.0,0.0,2336.0,23584.0,0.0,7296.0,63712.0,1312.0,121760.0,6220800
3,4224.0,0.0,18528.0,0.0,0.0,2176.0,11616.0,0.0,2208.0,18208.0,688.0,57648.0,6228000
4,0.0,0.0,32.0,0.0,0.0,1440.0,9792.0,0.0,4128.0,77824.0,1168.0,94384.0,6235200
...,...,...,...,...,...,...,...,...,...,...,...,...,...
382,0.0,0.0,0.0,22320.0,1632.0,576.0,0.0,0.0,416.0,8192.0,316.0,33452.0,8956800
383,0.0,0.0,96.0,720.0,64.0,352.0,0.0,0.0,0.0,15840.0,902.0,17974.0,8964000
384,0.0,896.0,288.0,0.0,1440.0,1024.0,0.0,0.0,0.0,12832.0,66.0,16546.0,8971200
385,0.0,32.0,64.0,0.0,352.0,448.0,0.0,0.0,0.0,7044.0,95.0,8035.0,8978400


In [9]:
staked_pool_size_set = staked_pool_size[staked_pool_size['slot'].isin(deposits_size_set['slot'])]
staked_pool_size_set.reset_index(inplace=True)
staked_pool_size_set = staked_pool_size_set.drop(columns=['index'])
staked_pool_size_set

,slot,1,100+,2-5,20-99,6-19,total
0,6206400.0,212736.0,16269280.0,269888.0,854176.0,404960.0,18011040.0
1,6213600.0,211904.0,16247936.0,269888.0,859072.0,405472.0,17994272.0
2,6220800.0,211936.0,16224864.0,270304.0,860512.0,405344.0,17972960.0
3,6228000.0,212128.0,16221888.0,270464.0,862848.0,405632.0,17972960.0
4,6235200.0,212256.0,16214080.0,270496.0,869632.0,406496.0,17972960.0
...,...,...,...,...,...,...,...
382,8956800.0,315680.0,29345728.0,317280.0,1412416.0,575936.0,31967040.0
383,8964000.0,316608.0,29385568.0,317248.0,1412864.0,577056.0,32009344.0
384,8971200.0,316736.0,29391936.0,317440.0,1416864.0,577056.0,32020032.0
385,8978400.0,314368.0,29405280.0,317632.0,1418976.0,577344.0,32033600.0


In [10]:
staked_category_set = staked_category[staked_category['slot'].isin(deposits_category_set['slot'])]
staked_category_set.reset_index(inplace=True)
staked_category_set = staked_category_set.drop(columns=['index'])
staked_category_set

,slot,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total
0,6206400.0,6325152.0,256.0,6499712.0,689824.0,837856.0,3658240.0,18011040.0
1,6213600.0,6293280.0,256.0,6505664.0,690048.0,838432.0,3666592.0,17994272.0
2,6220800.0,6247904.0,256.0,6506016.0,690048.0,840352.0,3688384.0,17972960.0
3,6228000.0,6218016.0,256.0,6501984.0,690400.0,841728.0,3720576.0,17972960.0
4,6235200.0,6188032.0,256.0,6502080.0,691776.0,844320.0,3746496.0,17972960.0
...,...,...,...,...,...,...,...,...
382,8956800.0,8177824.0,2429312.0,10604864.0,537984.0,1826464.0,8390592.0,31967040.0
383,8964000.0,8188000.0,2453920.0,10606112.0,538208.0,1823776.0,8399328.0,32009344.0
384,8971200.0,8193344.0,2465312.0,10602048.0,538272.0,1806080.0,8414976.0,32020032.0
385,8978400.0,8189504.0,2474016.0,10597536.0,538464.0,1806240.0,8427840.0,32033600.0


In [11]:
staked_pool_set = staked_pool[staked_pool['slot'].isin(deposits_pool_set['slot'])]
staked_pool_set.reset_index(inplace=True)
staked_pool_set = staked_pool_set.drop(columns=['index'])
staked_pool_set

,slot,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total
0,6206400.0,1131232.0,437664.0,2691776.0,0.0,1247520.0,74624.0,5677216.0,0.0,170304.0,6122848.0,457856.0,18011040.0
1,6213600.0,1131232.0,437664.0,2694848.0,0.0,1215264.0,75200.0,5681728.0,0.0,170720.0,6128640.0,458976.0,17994272.0
2,6220800.0,1131488.0,437664.0,2697056.0,0.0,1161760.0,77088.0,5682656.0,0.0,173920.0,6152352.0,458976.0,17972960.0
3,6228000.0,1131936.0,437664.0,2709760.0,0.0,1117344.0,78432.0,5690528.0,0.0,173920.0,6174496.0,458880.0,17972960.0
4,6235200.0,1131936.0,437664.0,2715424.0,0.0,1069312.0,79968.0,5696448.0,0.0,181152.0,6201952.0,459104.0,17972960.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
382,8956800.0,1125344.0,586112.0,4433696.0,1084352.0,757376.0,486912.0,9295712.0,483680.0,348064.0,12575232.0,790560.0,31967040.0
383,8964000.0,1125344.0,597280.0,4439072.0,1084352.0,758176.0,487232.0,9295712.0,480640.0,348064.0,12602560.0,790912.0,32009344.0
384,8971200.0,1125344.0,608256.0,4434528.0,1072800.0,758944.0,470304.0,9295712.0,479584.0,348064.0,12635680.0,790816.0,32020032.0
385,8978400.0,1125344.0,608416.0,4428800.0,1071200.0,759424.0,470624.0,9291072.0,479392.0,348064.0,12661152.0,790112.0,32033600.0


In [12]:
deposits_size_set.set_index('slot', inplace=True)
staked_pool_size_set.set_index('slot', inplace=True)
deposits_percentage_size = deposits_size_set.divide(staked_pool_size_set, fill_value=0) * 100
deposits_percentage_size.reset_index(inplace=True)
deposits_percentage_size

,slot,1,100+,2-5,20-99,6-19,total
0,6206400,0.030084,0.090477,0.047427,0.606901,0.126432,0.114419
1,6213600,0.090607,0.411129,0.284562,0.409744,0.157841,0.399683
2,6220800,0.362374,0.658348,0.331479,1.282957,0.552617,0.677462
3,6228000,0.211193,0.310198,0.260293,0.493250,0.473335,0.320749
4,6235200,0.120609,0.526061,0.319413,0.603474,0.669133,0.525144
...,...,...,...,...,...,...,...
382,8956800,0.070958,0.109958,0.141200,0.013594,0.055562,0.104645
383,8964000,0.050536,0.057899,0.151301,0.018119,0.011091,0.056152
384,8971200,0.394019,0.033104,0.070565,0.309416,0.166362,0.051674
385,8978400,0.061075,0.024496,0.120895,0.013531,0.011085,0.025083


In [13]:
deposits_category_set.set_index('slot', inplace=True)
staked_category_set.set_index('slot', inplace=True)
deposits_percentage_category = deposits_category_set.divide(staked_category_set, fill_value=0) * 100
deposits_percentage_category.reset_index(inplace=True)
deposits_percentage_category

,slot,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,total
0,6206400,0.071840,0.000000,0.086158,0.032472,0.068747,0.264171,0.114419
1,6213600,0.157628,0.000000,0.045007,0.027824,0.354948,1.524686,0.399683
2,6220800,0.585924,0.000000,0.442667,0.273604,0.403640,1.384671,0.677462
3,6228000,0.444129,0.000000,0.201538,0.194670,0.258516,0.360374,0.320749
4,6235200,0.484548,0.000000,0.201043,1.008419,0.170551,1.145390,0.525144
...,...,...,...,...,...,...,...,...
382,8956800,0.031695,0.964882,0.002980,0.011896,0.031536,0.077039,0.104645
383,8964000,0.008598,0.048901,0.008505,0.005946,0.457951,0.080768,0.056152
384,8971200,0.042571,0.031152,0.002132,0.000000,0.058469,0.130814,0.051674
385,8978400,0.025008,0.014228,0.009691,0.005943,0.024803,0.048981,0.025083


In [14]:
deposits_pool_set.set_index('slot', inplace=True)
staked_pool_set.set_index('slot', inplace=True)
deposits_percentage_pool = deposits_pool_set.divide(staked_pool_set, fill_value=0) * 100
deposits_percentage_pool.reset_index(inplace=True)
deposits_percentage_pool

,slot,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total
0,6206400,0.000000,0.000000,0.114125,NaN,0.053867,0.771870,0.079476,NaN,0.244269,0.172991,0.167738,0.114419
1,6213600,0.062233,0.000000,0.081934,NaN,0.000000,3.872340,0.016333,NaN,1.874414,1.009294,0.024402,0.399683
2,6220800,0.104641,0.000000,0.828162,NaN,0.000000,3.030303,0.415017,NaN,4.195032,1.035571,0.285854,0.677462
3,6228000,0.373166,0.000000,0.683751,NaN,0.000000,2.774378,0.204129,NaN,1.269549,0.294890,0.149930,0.320749
4,6235200,0.000000,0.000000,0.001178,NaN,0.000000,1.800720,0.171897,NaN,2.278749,1.254831,0.254409,0.525144
...,...,...,...,...,...,...,...,...,...,...,...,...,...
382,8956800,0.000000,0.000000,0.000000,2.058372,0.215481,0.118297,0.000000,0.0,0.119518,0.065144,0.039972,0.104645
383,8964000,0.000000,0.000000,0.002163,0.066399,0.008441,0.072245,0.000000,0.0,0.000000,0.125689,0.114046,0.056152
384,8971200,0.000000,0.147306,0.006494,0.000000,0.189737,0.217732,0.000000,0.0,0.000000,0.101554,0.008346,0.051674
385,8978400,0.000000,0.005260,0.001445,0.000000,0.046351,0.095193,0.000000,0.0,0.000000,0.055635,0.012024,0.025083


In [15]:
columns = ['1', '100+', '2-5', '20-99', '6-19', 'total']
for col in columns:
    print(deposits_percentage_size[col].mean())

0.24308380009907732
0.28843297974599247
0.14796806601097864
0.20623054091848853
0.19565206871390312
0.2803493303784511


In [16]:
deposits_percentage_size

,slot,1,100+,2-5,20-99,6-19,total
0,6206400,0.030084,0.090477,0.047427,0.606901,0.126432,0.114419
1,6213600,0.090607,0.411129,0.284562,0.409744,0.157841,0.399683
2,6220800,0.362374,0.658348,0.331479,1.282957,0.552617,0.677462
3,6228000,0.211193,0.310198,0.260293,0.493250,0.473335,0.320749
4,6235200,0.120609,0.526061,0.319413,0.603474,0.669133,0.525144
...,...,...,...,...,...,...,...
382,8956800,0.070958,0.109958,0.141200,0.013594,0.055562,0.104645
383,8964000,0.050536,0.057899,0.151301,0.018119,0.011091,0.056152
384,8971200,0.394019,0.033104,0.070565,0.309416,0.166362,0.051674
385,8978400,0.061075,0.024496,0.120895,0.013531,0.011085,0.025083


In [17]:
import numpy as np
import pandas as pd
from scipy import stats

# Assuming deposits_percentage_size and rewards_size DataFrames are already defined

# Drop the epoch column from rewards_size
# rewards_size = rewards_size.drop(columns=['epoch'])

# Calculate the percent change of APY
rewards_size_pct_change = rewards_size.set_index('slot').pct_change().reset_index()

# Merge the two DataFrames on the slot column
price_elasticity_size = pd.merge(deposits_percentage_size, rewards_size_pct_change, on='slot')
price_elasticity_size

,slot,1_x,100+_x,2-5_x,20-99_x,6-19_x,total_x,1_y,100+_y,2-5_y,20-99_y,6-19_y,total_y
0,6840000,0.060042,0.027694,0.057904,0.003327,0.201758,0.030990,NaN,NaN,NaN,NaN,NaN,NaN
1,6847200,0.180180,0.154382,0.150393,0.046138,0.100966,0.148466,-0.011630,-0.001953,-0.000401,0.019836,0.010957,0.003430
2,6854400,0.195254,0.248938,0.034706,0.249975,0.345299,0.247666,-0.008151,-0.000911,-0.004860,0.005972,-0.006411,-0.002773
3,6861600,0.150331,0.481460,0.208309,0.092199,0.331068,0.453516,-0.001485,-0.004306,0.002392,0.011041,0.031263,0.007832
4,6868800,0.240819,0.348897,0.115794,0.309628,0.122704,0.338247,-0.010730,-0.000065,0.005750,-0.013196,-0.036917,-0.011187
...,...,...,...,...,...,...,...,...,...,...,...,...,...
294,8956800,0.070958,0.109958,0.141200,0.013594,0.055562,0.104645,-0.011943,0.002562,-0.022963,-0.006187,0.017135,-0.004255
295,8964000,0.050536,0.057899,0.151301,0.018119,0.011091,0.056152,0.057568,0.002643,0.010066,0.001762,-0.002399,0.013467
296,8971200,0.394019,0.033104,0.070565,0.309416,0.166362,0.051674,-0.048042,0.000640,-0.051941,0.012107,0.007081,-0.015751
297,8978400,0.061075,0.024496,0.120895,0.013531,0.011085,0.025083,0.002260,0.003030,0.027027,-0.007463,-0.002045,0.004151


In [18]:
# Calculate the percent change of APY
rewards_size_pct_change = rewards_size.set_index('slot').pct_change().reset_index()

# Merge the two DataFrames on the slot column
price_elasticity_size = pd.merge(deposits_percentage_size, rewards_size_pct_change, on='slot')

# Drop rows with infinite values
price_elasticity_size.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_size['elasticity_total'] = price_elasticity_size['total_x'] / price_elasticity_size['total_y']
price_elasticity_size['elasticity_total'] = price_elasticity_size['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_size['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column except 'total'
columns = ['1', '2-5', '6-19', '20-99', '100+']
for col in columns:
    elasticity_col_name = f'elasticity_{col}'
    price_elasticity_size[elasticity_col_name] = price_elasticity_size[f'{col}_x'] / price_elasticity_size[f'{col}_y']
    
    # Replace infinite values with NaN
    price_elasticity_size[elasticity_col_name] = price_elasticity_size[elasticity_col_name].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_size[elasticity_col_name].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_size)

Elasticity Analysis Results:
total: Mean Elasticity = 56.95055169981298, t(Mean) = -, SD = 685.5405635051965, N = 298, p-value = -
1: Mean Elasticity = -3.3039934157647464, t(Mean) = -1.4975041895816252, SD = 111.76963499596249, N = 298, p-value = 0.135270575361876
2-5: Mean Elasticity = -1.1851163659320814, t(Mean) = -1.4386667014566383, SD = 129.01296113180987, N = 298, p-value = 0.15122842096027062
6-19: Mean Elasticity = -4.886619258571938, t(Mean) = -1.4298629680766919, SD = 295.60513984594314, N = 298, p-value = 0.15352968664765396
20-99: Mean Elasticity = 74.52754998432854, t(Mean) = 0.2778098566009677, SD = 850.2655853650531, N = 298, p-value = 0.7812593367414394
100+: Mean Elasticity = 852.7968946602506, t(Mean) = 0.9973781021636637, SD = 13757.483653632828, N = 298, p-value = 0.3193889018022115
        slot       1_x    100+_x     2-5_x   20-99_x    6-19_x   total_x  \
0    6840000  0.060042  0.027694  0.057904  0.003327  0.201758  0.030990   
1    6847200  0.180180  0.154382

In [19]:
# Calculate the percent change of APY
rewards_category_pct_change = rewards_category.set_index('slot').pct_change().reset_index()

# Merge the two DataFrames on the slot column
price_elasticity_category = pd.merge(deposits_percentage_category, rewards_category_pct_change, on='slot')

# Drop rows with infinite values
price_elasticity_category.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_category['elasticity_total'] = price_elasticity_category['total_x'] / price_elasticity_category['total_y']
price_elasticity_category['elasticity_total'] = price_elasticity_category['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_category['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column
columns = [col for col in deposits_percentage_category.columns if col != 'slot']
for col in columns:
    elasticity_col_name = f'elasticity_{col}'
    price_elasticity_category[elasticity_col_name] = price_elasticity_category[f'{col}_x'] / price_elasticity_category[f'{col}_y']
    
    # Replace infinite values with NaN
    price_elasticity_category[elasticity_col_name] = price_elasticity_category[elasticity_col_name].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_category[elasticity_col_name].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_category)

Elasticity Analysis Results:
total: Mean Elasticity = -5.052694370458872, t(Mean) = 0.0, SD = 381.810976879006, N = 298, p-value = 1.0
CEX: Mean Elasticity = 41.73871517685295, t(Mean) = 1.5189533995934088, SD = 370.145234498128, N = 298, p-value = 0.12930679332092185
Liquid Restaking: Mean Elasticity = -189.67977056067295, t(Mean) = -1.3140712624649387, SD = 2395.165744665222, N = 298, p-value = 0.1897872699084867
Liquid Staking: Mean Elasticity = -4.019621414317599, t(Mean) = 0.03730910028619418, SD = 287.5771979172491, N = 298, p-value = 0.9702520395941407
Solo Stakers: Mean Elasticity = -1.6572211811730988, t(Mean) = 0.1530361150016687, SD = 30.331591625602158, N = 298, p-value = 0.8784724203947549
Staking Pools: Mean Elasticity = -22.130177927941084, t(Mean) = -0.6212622105381185, SD = 281.76630572962006, N = 298, p-value = 0.5346861554429372
Unidentified: Mean Elasticity = -43.95330949311103, t(Mean) = -0.8421857165789192, SD = 700.0071676923213, N = 298, p-value = 0.400122269350

In [20]:
# Calculate the percent change of APY
rewards_pool_pct_change = rewards_pool.set_index('slot').pct_change().reset_index()

# Merge the two DataFrames on the slot column
price_elasticity_pool = pd.merge(deposits_percentage_pool, rewards_pool_pct_change, on='slot')

# Drop rows with infinite values
price_elasticity_pool.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate elasticity for 'total' first
price_elasticity_pool['elasticity_total'] = price_elasticity_pool['total_x'] / price_elasticity_pool['total_y']
price_elasticity_pool['elasticity_total'] = price_elasticity_pool['elasticity_total'].replace([np.inf, -np.inf], np.nan)

# Initialize dictionaries to store elasticity values, standard deviations, t-statistics, p-values, and number of valid rows (N)
elasticity = {}
standard_deviations = {}
t_statistics = {}
p_values = {}
valid_counts = {}

# Calculate elasticity for the 'total' column
valid_total_elasticity = price_elasticity_pool['elasticity_total'].dropna()
elasticity['total'] = valid_total_elasticity.mean()
standard_deviations['total'] = valid_total_elasticity.std()
t_statistics['total'] = '-'
p_values['total'] = '-'
valid_counts['total'] = len(valid_total_elasticity)

# Calculate elasticity for each column
columns = ['Lido', 'Coinbase', 'Binance', 'Rocketpool', 'Kraken', 'OKX', 'Bitcoin Suisse', 'Ledger Live', 'Ether.Fi', 'Mantle', 'Other Stakers']
for col in columns:
    elasticity_col_name = f'elasticity_{col}'
    price_elasticity_pool[elasticity_col_name] = price_elasticity_pool[f'{col}_x'] / price_elasticity_pool[f'{col}_y']
    
    # Replace infinite values with NaN
    price_elasticity_pool[elasticity_col_name] = price_elasticity_pool[elasticity_col_name].replace([np.inf, -np.inf], np.nan)
    
    # Drop NaN values for calculation purposes
    valid_elasticity = price_elasticity_pool[elasticity_col_name].dropna()
    
    elasticity[col] = valid_elasticity.mean()
    standard_deviations[col] = valid_elasticity.std()
    
    # Perform a two-sample t-test against the 'total' elasticity
    t_stat, p_value = stats.ttest_ind(valid_elasticity, valid_total_elasticity, equal_var=False)
    t_statistics[col] = t_stat
    p_values[col] = p_value
    
    # Store the number of valid rows
    valid_counts[col] = len(valid_elasticity)

# Print the results in the requested format
print("Elasticity Analysis Results:")
print(f'total: Mean Elasticity = {elasticity["total"]}, t(Mean) = {t_statistics["total"]}, SD = {standard_deviations["total"]}, N = {valid_counts["total"]}, p-value = {p_values["total"]}')
for col in columns:
    print(f'{col}: Mean Elasticity = {elasticity[col]}, t(Mean) = {t_statistics[col]}, SD = {standard_deviations[col]}, N = {valid_counts[col]}, p-value = {p_values[col]}')

# Display the DataFrame with elasticity columns
print(price_elasticity_pool)

Elasticity Analysis Results:
total: Mean Elasticity = -26.667085686715538, t(Mean) = -, SD = 1136.8107965845074, N = 298, p-value = -
Lido: Mean Elasticity = 0.5916252843943234, t(Mean) = 0.4032830342128578, SD = 262.92167437466344, N = 298, p-value = 0.6870022328006011
Coinbase: Mean Elasticity = 26.13153119712881, t(Mean) = 0.6039165624313845, SD = 992.6825573717543, N = 298, p-value = 0.5461338651365791
Binance: Mean Elasticity = 4.872968427060952, t(Mean) = 0.4761205276839597, SD = 123.93059373254253, N = 298, p-value = 0.634330439889119
Rocketpool: Mean Elasticity = 0.39772009922728707, t(Mean) = 0.4103898690592721, SD = 61.190397875452604, N = 298, p-value = 0.6818141672036259
Kraken: Mean Elasticity = 48.795262216970656, t(Mean) = 1.0008639026871746, SD = 633.808811120906, N = 298, p-value = 0.31741273075093235
OKX: Mean Elasticity = -21.245341501426672, t(Mean) = 0.08006430124693507, SD = 272.36506087266434, N = 298, p-value = 0.9362345020460281
Bitcoin Suisse: Mean Elasticity 

/var/folders/mb/5hm6pgrs3zj_1m_kgvpt40jw0000gn/T/ipykernel_24014/1779073167.py:2: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  rewards_pool_pct_change = rewards_pool.set_index('slot').pct_change().reset_index()
